# Debug `japanize_matplotlib`

This notebook inspects what `import japanize_matplotlib` changes in Matplotlib. It is designed to compare that behavior with project-local `.fonts` plus `matplotlibrc`.

For the clearest results, restart the kernel before running this notebook. Once `japanize_matplotlib` is imported, it mutates Matplotlib's font manager and rcParams in the current kernel.

In [ ]:
import importlib.metadata as metadata
import importlib.util
import pathlib

spec = importlib.util.find_spec("japanize_matplotlib")
package_dir = pathlib.Path(spec.origin).parent
font_dir = package_dir / "fonts"

print("version:", metadata.version("japanize-matplotlib"))
print("package:", package_dir)
print("font dir:", font_dir)
print("font files:")
for path in sorted(font_dir.iterdir()):
    print(" ", path.name, path.stat().st_size)

In [ ]:
source_path = package_dir / "japanize_matplotlib.py"
source = source_path.read_text(encoding="utf-8")

print(source_path)
for line in source.splitlines():
    if any(keyword in line for keyword in ["FONT_NAME", "FONT_TTF", "addfont", "matplotlib.rc", "japanize()"]):
        print(line)

The next cells use isolated Python subprocesses. This lets us compare Matplotlib before and after `import japanize_matplotlib` without contamination from the current kernel.

In [ ]:
import os
import subprocess
import sys
import textwrap
from pathlib import Path


def run_python(label, code):
    result = subprocess.run(
        [sys.executable, "-c", textwrap.dedent(code)],
        cwd=Path.cwd(),
        env=os.environ.copy(),
        text=True,
        capture_output=True,
        check=False,
    )
    print(f"===== {label} =====")
    print("returncode:", result.returncode)
    if result.stdout:
        print("--- stdout ---")
        print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
    return result

In [ ]:
run_python(
    "without japanize_matplotlib",
    r'''
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.font_manager as fm
    import matplotlib.pyplot as plt

    print("matplotlibrc:", matplotlib.matplotlib_fname())
    print("font.family:", plt.rcParams["font.family"])
    print("font.sans-serif:", plt.rcParams["font.sans-serif"][:5])

    found = [font for font in fm.fontManager.ttflist if font.name == "IPAexGothic"]
    print("IPAexGothic in ttflist:", len(found))

    try:
        print("findfont IPAexGothic:", fm.findfont("IPAexGothic", fallback_to_default=False))
    except Exception as exc:
        print("findfont IPAexGothic failed:", type(exc).__name__, exc)

    fig, ax = plt.subplots()
    ax.plot([1, 2, 3], [10, 20, 15])
    ax.set_title("日本語タイトル without japanize")
    fig.savefig("/tmp/mpl_without_japanize.png")
    print("saved: /tmp/mpl_without_japanize.png")
    ''',
)

In [ ]:
run_python(
    "with japanize_matplotlib",
    r'''
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.font_manager as fm
    import matplotlib.pyplot as plt

    print("before import font.family:", plt.rcParams["font.family"])
    import japanize_matplotlib
    print("after import font.family:", plt.rcParams["font.family"])
    print("matplotlibrc:", matplotlib.matplotlib_fname())

    found = [font for font in fm.fontManager.ttflist if font.name == "IPAexGothic"]
    print("IPAexGothic in ttflist:", len(found))
    for font in found[:3]:
        print("found:", font.name, font.fname)

    print("findfont IPAexGothic:", fm.findfont("IPAexGothic", fallback_to_default=False))

    fig, ax = plt.subplots()
    ax.plot([1, 2, 3], [10, 20, 15])
    ax.set_title("日本語タイトル with japanize")
    fig.savefig("/tmp/mpl_with_japanize.png")
    print("saved: /tmp/mpl_with_japanize.png")
    ''',
)

The next cells mutate the current kernel. Run them when you are ready to make the notebook itself use `japanize_matplotlib`.

In [ ]:
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

print("before:", plt.rcParams["font.family"])

import japanize_matplotlib

print("after:", plt.rcParams["font.family"])
print("matplotlibrc:", matplotlib.matplotlib_fname())
print("findfont:", fm.findfont("IPAexGothic", fallback_to_default=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot([1, 2, 3, 4], [10, 18, 13, 22], marker="o")
ax.set_title("japanize_matplotlib による日本語描画")
ax.set_xlabel("回数")
ax.set_ylabel("値")
ax.grid(True)
plt.show()